# 08 - Test-Representation Similarity Analysis

This notebook compares the image and spectra representations used for similarity and retrieval. It first calculates validation image-embedding similarities for the frozen and fine-tuned CNN iterations, then adds the selected spectra representation used by the composition-estimation pipeline.

The analysis uses the validation split because the representation comparison belongs to model development and interpretation.


## Setup

The setup matches the active CNN notebooks: grayscale `Stereo / T / C+M+P` image inputs are passed to DenseNet201 as RGB-shaped tensors, because the ImageNet-pretrained model expects three channels. The spectra part uses the selected `Benchtop_N / N/A` representation.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import random
from typing import Iterable

local_torch_home = Path.home() / ".cache" / "torch"
local_densenet_weights = local_torch_home / "hub" / "checkpoints" / "densenet201-c1103571.pth"
if local_densenet_weights.exists():
    os.environ.setdefault("TORCH_HOME", str(local_torch_home))
else:
    os.environ.setdefault("TORCH_HOME", "/tmp/mdpp_v2_notebook_cache/torch")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mdpp_v2_notebook_cache/matplotlib")

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import DenseNet201_Weights, densenet201
from IPython.display import Markdown, display

The constants intentionally match notebooks `03`, `05`, and `06`. The only split loaded here is validation.

In [ ]:
RANDOM_SEED = 42
BATCH_SIZE = 16
IMAGE_MODALITY = "Stereo"
IMAGE_INSTRUMENT = "T"
SELECTED_SETTING = "Colour + mono + polarized grayscale"
FROZEN_CNN_CHECKPOINT_NAME = "cnn_frozen_densenet201_linear_head.pt"
FINETUNED_CNN_CHECKPOINT_NAME = "cnn_finetuned_densenet201_denseblock4_head.pt"

FROZEN_MODEL_LABEL = "iteration_1_frozen_densenet201"
FINETUNED_MODEL_LABEL = "iteration_2_finetuned_denseblock4"
MODEL_LABELS = [FROZEN_MODEL_LABEL, FINETUNED_MODEL_LABEL]
MODEL_DISPLAY_NAMES = {
    FROZEN_MODEL_LABEL: "Iteration 1: frozen DenseNet201",
    FINETUNED_MODEL_LABEL: "Iteration 2: fine-tuned denseblock4",
}
MODEL_PLOT_LABELS = {
    FROZEN_MODEL_LABEL: "Frozen",
    FINETUNED_MODEL_LABEL: "Fine-tuned",
}
MODEL_COLOURS = {
    FROZEN_MODEL_LABEL: "#2563eb",
    FINETUNED_MODEL_LABEL: "#dc2626",
}

CLASS_NAMES = [
    "Cotton",
    "Natural polyamide",
    "Polyester / recycled polyester / PET",
    "Regenerated cellulosic",
    "Synthetic polyamide / nylon",
]
LABEL_TO_INDEX = {label: index for index, label in enumerate(CLASS_NAMES)}
INDEX_TO_LABEL = {index: label for label, index in LABEL_TO_INDEX.items()}

FEATURE_SOURCES = [
    {"feature_source": "C_gray", "formats": ("C",), "preprocessing": "L_to_RGB"},
    {"feature_source": "M_gray", "formats": ("M",), "preprocessing": "L_to_RGB"},
    {"feature_source": "P_gray", "formats": ("P",), "preprocessing": "L_to_RGB"},
]
PAIR_GROUP_ORDER = ["Same fabric", "Same class, different fabric", "Different class"]
PAIR_GROUP_LABELS = ["Same fabric", "Same class,\ndifferent fabric", "Different class"]
SPLITS = ["val"]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))


def select_torch_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    print("CUDA is not available; using CPU for this inference notebook.")
    return torch.device("cpu")


DEVICE = select_torch_device()
PIN_MEMORY = DEVICE.type == "cuda"
DEVICE

## 1. Paths and Validation Split

The notebook resolves the project root from the current working directory and reads only the validation split artifacts.

In [ ]:
def resolve_mdpp_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "mdpp_v2" / "data_splits").exists():
            return candidate / "mdpp_v2"
        if candidate.name == "mdpp_v2" and (candidate / "data_splits").exists():
            return candidate
    raise FileNotFoundError("Could not resolve mdpp_v2 root from current working directory.")


MDPP_ROOT = resolve_mdpp_root()
DATA_SPLITS_DIR = MDPP_ROOT / "data_splits"
ARTIFACTS_DIR = MDPP_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

FROZEN_CNN_CHECKPOINT_PATH = ARTIFACTS_DIR / FROZEN_CNN_CHECKPOINT_NAME
FINETUNED_CNN_CHECKPOINT_PATH = ARTIFACTS_DIR / FINETUNED_CNN_CHECKPOINT_NAME
BOXPLOT_PATH = ARTIFACTS_DIR / "image_embedding_similarity_boxplot.png"
SUMMARY_CSV_PATH = ARTIFACTS_DIR / "image_embedding_similarity_summary.csv"
PAIRWISE_CSV_PATH = ARTIFACTS_DIR / "pairwise_image_embedding_similarities.csv"

split_sample_tables = {split: pd.read_csv(DATA_SPLITS_DIR / split / "samples.csv") for split in SPLITS}
split_manifest_tables = {split: pd.read_csv(DATA_SPLITS_DIR / split / "measurements_manifest.csv") for split in SPLITS}

print(f"Using device: {DEVICE}")
print(f"Torch home: {os.environ['TORCH_HOME']}")

## 2. Build the Validation Image Table

Only CNN-eligible validation fabrics are included. The selected image representation uses colour, monochromatic, and polarised image files after grayscale conversion.

In [ ]:
def eligible_samples_for_split(split: str) -> pd.DataFrame:
    samples = split_sample_tables[split].copy()
    eligible = samples[samples["cnn_label"].ne("EXCLUDED_CNN")].copy()
    eligible = eligible[["Sample ID", "classification_label", "cnn_label", "split"]]
    eligible = eligible.rename(columns={"Sample ID": "sample_id"})
    eligible["label_index"] = eligible["classification_label"].map(LABEL_TO_INDEX)
    if eligible["label_index"].isna().any():
        missing = sorted(eligible.loc[eligible["label_index"].isna(), "classification_label"].dropna().unique())
        raise ValueError(f"Unexpected retained CNN labels: {missing}")
    return eligible


def build_source_table(split: str, feature_source: str, formats: Iterable[str], preprocessing: str) -> pd.DataFrame:
    manifest = split_manifest_tables[split].copy()
    eligible = eligible_samples_for_split(split)
    mask = (
        manifest["modality_group"].eq("image")
        & manifest["modality"].eq(IMAGE_MODALITY)
        & manifest["instrument"].eq(IMAGE_INSTRUMENT)
        & manifest["data_format"].isin(tuple(formats))
    )
    table = manifest.loc[mask].merge(eligible, on="sample_id", how="inner", suffixes=("", "_sample"))
    table["path"] = table["final_path"].map(Path)
    table["image_path"] = table["path"]
    table["feature_source"] = feature_source
    table["preprocessing"] = preprocessing
    table["setting"] = SELECTED_SETTING
    table["source_split"] = split
    if table["cnn_label"].eq("EXCLUDED_CNN").any():
        raise ValueError("EXCLUDED_CNN rows entered the validation image table.")
    missing_files = [path for path in table["path"] if not path.exists()]
    if missing_files:
        raise FileNotFoundError(f"Missing image files, first examples: {missing_files[:3]}")
    return table.sort_values(["sample_id", "feature_source", "replicate", "filename"]).reset_index(drop=True)

In [ ]:
source_tables = []
for source_spec in FEATURE_SOURCES:
    source_tables.append(
        build_source_table(
            split="val",
            feature_source=source_spec["feature_source"],
            formats=source_spec["formats"],
            preprocessing=source_spec["preprocessing"],
        )
    )

val_table = (
    pd.concat(source_tables, ignore_index=True)
    .sort_values(["sample_id", "feature_source", "replicate", "filename"])
    .reset_index(drop=True)
)
val_table["row_index"] = np.arange(len(val_table))
val_table["image_id"] = val_table.apply(
    lambda row: f"val:{int(row['row_index']):04d}:{row['feature_source']}:{row['filename']}",
    axis=1,
)

validation_input_summary = pd.DataFrame([
    {
        "split": "validation",
        "images": len(val_table),
        "fabrics": val_table["sample_id"].nunique(),
        "classes": val_table["classification_label"].nunique(),
        "image_formats": "+".join(sorted(val_table["data_format"].unique())),
        "preprocessing": "L -> RGB grayscale",
    }
])
validation_input_summary

## 3. CNN Embedding Models

Both saved checkpoints are loaded for status and compatibility checks. No training happens in this notebook.

In [ ]:
weights = DenseNet201_Weights.DEFAULT
weights_transform = weights.transforms()


class DenseNetEmbeddingExtractor(nn.Module):
    def __init__(self, model: nn.Module):
        super().__init__()
        self.features = model.features

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.features(images)
        out = F.relu(features, inplace=False)
        out = F.adaptive_avg_pool2d(out, (1, 1))
        return torch.flatten(out, 1)


def load_checkpoint(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {path}")
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    if checkpoint.get("class_names") != CLASS_NAMES:
        raise ValueError(f"Checkpoint class names do not match this notebook: {path}")
    return checkpoint


def strip_state_dict_prefix(state_dict: dict[str, torch.Tensor], prefix: str) -> dict[str, torch.Tensor]:
    return {key.removeprefix(prefix): value for key, value in state_dict.items() if key.startswith(prefix)}


def freeze_for_inference(model: nn.Module) -> nn.Module:
    for parameter in model.parameters():
        parameter.requires_grad = False
    return model.to(DEVICE).eval()

In [ ]:
def make_frozen_embedding_model() -> tuple[nn.Module, dict]:
    checkpoint = load_checkpoint(FROZEN_CNN_CHECKPOINT_PATH)
    model = densenet201(weights=weights)
    feature_dim = model.classifier.in_features
    model.classifier = nn.Linear(feature_dim, len(CLASS_NAMES))
    model.classifier.load_state_dict(checkpoint["state_dict"])
    extractor = freeze_for_inference(DenseNetEmbeddingExtractor(model))
    return extractor, {
        "model": FROZEN_MODEL_LABEL,
        "display_name": MODEL_DISPLAY_NAMES[FROZEN_MODEL_LABEL],
        "checkpoint": FROZEN_CNN_CHECKPOINT_PATH.name,
        "status": "loaded",
        "checkpoint_best_epoch": checkpoint.get("best_epoch"),
        "feature_dim": feature_dim,
        "trainable_parameters": sum(parameter.numel() for parameter in extractor.parameters() if parameter.requires_grad),
    }


def make_finetuned_embedding_model() -> tuple[nn.Module, dict]:
    checkpoint = load_checkpoint(FINETUNED_CNN_CHECKPOINT_PATH)
    model = densenet201(weights=weights)
    feature_dim = model.classifier.in_features
    model.classifier = nn.Linear(feature_dim, len(CLASS_NAMES))
    tail_state = checkpoint["tail_state_dict"]
    model.features.denseblock4.load_state_dict(strip_state_dict_prefix(tail_state, "denseblock4."))
    model.features.norm5.load_state_dict(strip_state_dict_prefix(tail_state, "norm5."))
    model.classifier.load_state_dict(strip_state_dict_prefix(tail_state, "classifier."))
    extractor = freeze_for_inference(DenseNetEmbeddingExtractor(model))
    return extractor, {
        "model": FINETUNED_MODEL_LABEL,
        "display_name": MODEL_DISPLAY_NAMES[FINETUNED_MODEL_LABEL],
        "checkpoint": FINETUNED_CNN_CHECKPOINT_PATH.name,
        "status": "loaded",
        "checkpoint_best_epoch": checkpoint.get("best_epoch"),
        "feature_dim": feature_dim,
        "trainable_parameters": sum(parameter.numel() for parameter in extractor.parameters() if parameter.requires_grad),
    }

The checkpoint-status table makes the loaded model state visible without showing raw checkpoint internals.

In [ ]:
embedding_model_factories = [
    (FROZEN_MODEL_LABEL, make_frozen_embedding_model),
    (FINETUNED_MODEL_LABEL, make_finetuned_embedding_model),
]
embedding_models = []
checkpoint_rows = []
for model_label, factory in embedding_model_factories:
    model, metadata = factory()
    embedding_models.append((model_label, model))
    checkpoint_rows.append(metadata)

checkpoint_status = pd.DataFrame(checkpoint_rows)
checkpoint_status[["display_name", "checkpoint", "status", "checkpoint_best_epoch", "feature_dim", "trainable_parameters"]]

## 4. Extract Image-Level Embeddings

Embeddings are extracted per image file. They are not averaged by fabric, because the first question is whether repeated images from the same fabric are close to each other.

In [ ]:
def preprocess_image_from_row(row: pd.Series) -> torch.Tensor:
    with Image.open(row["path"]) as image:
        # DenseNet201 expects three channels. The image content remains grayscale.
        if row["preprocessing"] == "L_to_RGB":
            image = image.convert("L").convert("RGB")
        elif row["preprocessing"] == "RGB":
            image = image.convert("RGB")
        else:
            raise ValueError(f"Unknown preprocessing mode: {row['preprocessing']}")
        return weights_transform(image)


class EmbeddingImageDataset(Dataset):
    def __init__(self, table: pd.DataFrame):
        self.table = table.reset_index(drop=True).copy()

    def __len__(self) -> int:
        return len(self.table)

    def __getitem__(self, idx: int):
        row = self.table.iloc[idx]
        return preprocess_image_from_row(row), int(row["row_index"])


@torch.no_grad()
def extract_image_embeddings(model: nn.Module, table: pd.DataFrame, model_label: str) -> tuple[pd.DataFrame, np.ndarray]:
    dataset = EmbeddingImageDataset(table)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)
    metadata = table.set_index("row_index")
    rows = []
    embedding_batches = []

    for images, row_indices in loader:
        embeddings = model(images.to(DEVICE, non_blocking=PIN_MEMORY)).detach().cpu().numpy()
        embedding_batches.append(embeddings)
        for row_index in row_indices.numpy():
            source = metadata.loc[int(row_index)]
            rows.append({
                "model": model_label,
                "row_index": int(row_index),
                "image_id": source["image_id"],
                "filename": source["filename"],
                "fabric_id": source["sample_id"],
                "class_label": source["classification_label"],
                "label_index": int(source["label_index"]),
                "feature_source": source["feature_source"],
                "data_format": source["data_format"],
                "source_split": source["source_split"],
            })
    return pd.DataFrame(rows), np.vstack(embedding_batches).astype("float32")


def l2_normalise(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("Found zero-norm embedding.")
    return (matrix / norms).astype("float32")

In [ ]:
embedding_frames = []
embedding_matrices = {}
normalised_embedding_matrices = {}

for model_label, model in embedding_models:
    frame, matrix = extract_image_embeddings(model, val_table, model_label)
    embedding_frames.append(frame)
    embedding_matrices[model_label] = matrix
    normalised_embedding_matrices[model_label] = l2_normalise(matrix)
    print(f"{model_label}: extracted {len(frame)} image embeddings for {frame['fabric_id'].nunique()} fabrics.")

image_embedding_frame = pd.concat(embedding_frames, ignore_index=True)
embedding_extraction_summary = checkpoint_status.merge(
    image_embedding_frame.groupby("model", as_index=False).agg(
        image_embeddings=("image_id", "size"),
        fabrics=("fabric_id", "nunique"),
        classes=("class_label", "nunique"),
    ),
    on="model",
    how="left",
)
embedding_extraction_summary[["display_name", "image_embeddings", "fabrics", "classes", "feature_dim", "trainable_parameters"]]

## 5. Pairwise Cosine Similarities

Before cosine similarity is calculated, embeddings are L2-normalised so each vector has unit length. Cosine similarity is then the dot product between normalised embeddings.

In [ ]:
def pairwise_similarities_for_model(model_label: str) -> pd.DataFrame:
    frame = image_embedding_frame[image_embedding_frame["model"].eq(model_label)].reset_index(drop=True)
    matrix = normalised_embedding_matrices[model_label]
    row_indices, col_indices = np.triu_indices(len(frame), k=1)

    image_ids = frame["image_id"].to_numpy()
    fabric_ids = frame["fabric_id"].to_numpy()
    labels = frame["class_label"].to_numpy()
    filenames = frame["filename"].to_numpy()
    feature_sources = frame["feature_source"].to_numpy()

    same_fabric = fabric_ids[row_indices] == fabric_ids[col_indices]
    same_class_different_fabric = (~same_fabric) & (labels[row_indices] == labels[col_indices])
    different_class = labels[row_indices] != labels[col_indices]
    pair_group = np.select(
        [same_fabric, same_class_different_fabric, different_class],
        PAIR_GROUP_ORDER,
        default="UNASSIGNED",
    )
    if np.any(pair_group == "UNASSIGNED"):
        raise ValueError("Some image pairs were not assigned to a pair group.")

    cosine_similarity = np.sum(matrix[row_indices] * matrix[col_indices], axis=1)
    output = pd.DataFrame({
        "iteration": MODEL_DISPLAY_NAMES[model_label],
        "model": model_label,
        "pair_group": pd.Categorical(pair_group, categories=PAIR_GROUP_ORDER, ordered=True),
        "image_id_a": image_ids[row_indices],
        "image_id_b": image_ids[col_indices],
        "filename_a": filenames[row_indices],
        "filename_b": filenames[col_indices],
        "fabric_id_a": fabric_ids[row_indices],
        "fabric_id_b": fabric_ids[col_indices],
        "class_a": labels[row_indices],
        "class_b": labels[col_indices],
        "feature_source_a": feature_sources[row_indices],
        "feature_source_b": feature_sources[col_indices],
        "cosine_similarity": cosine_similarity.astype(float),
    })
    return output.sort_values(["model", "pair_group", "image_id_a", "image_id_b"]).reset_index(drop=True)


pairwise_image_similarities = pd.concat(
    [pairwise_similarities_for_model(model_label) for model_label in MODEL_LABELS],
    ignore_index=True,
)
pair_counts = (
    pairwise_image_similarities.groupby(["iteration", "pair_group"], observed=False)
    .size()
    .rename("pairs")
    .reset_index()
)
pair_counts

## 6. Image Similarity Summary and Figure

The summary table, raw pair table, and boxplot are saved for reporting and reproducibility.

In [ ]:
def q1(values: pd.Series) -> float:
    return float(values.quantile(0.25))


def q3(values: pd.Series) -> float:
    return float(values.quantile(0.75))


image_embedding_similarity_summary = (
    pairwise_image_similarities.groupby(["iteration", "model", "pair_group"], observed=False)["cosine_similarity"]
    .agg(
        pairs="count",
        mean_cosine_similarity="mean",
        median_cosine_similarity="median",
        min_cosine_similarity="min",
        max_cosine_similarity="max",
        q1_cosine_similarity=q1,
        q3_cosine_similarity=q3,
    )
    .reset_index()
    .sort_values(["model", "pair_group"])
    .reset_index(drop=True)
)

image_embedding_similarity_summary.to_csv(SUMMARY_CSV_PATH, index=False)
pairwise_image_similarities.to_csv(PAIRWISE_CSV_PATH, index=False)
image_embedding_similarity_summary

The boxplot compares the three pair groups for both CNN iterations. Higher cosine similarity means the image embeddings are closer in direction.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
base_positions = np.arange(len(PAIR_GROUP_ORDER), dtype=float)
offsets = {FROZEN_MODEL_LABEL: -0.18, FINETUNED_MODEL_LABEL: 0.18}
box_width = 0.30

for model_label in MODEL_LABELS:
    model_frame = pairwise_image_similarities[pairwise_image_similarities["model"].eq(model_label)]
    values = [
        model_frame.loc[model_frame["pair_group"].eq(pair_group), "cosine_similarity"].to_numpy()
        for pair_group in PAIR_GROUP_ORDER
    ]
    box = ax.boxplot(
        values,
        positions=base_positions + offsets[model_label],
        widths=box_width,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "white", "linewidth": 1.5},
    )
    for patch in box["boxes"]:
        patch.set_facecolor(MODEL_COLOURS[model_label])
        patch.set_alpha(0.78)
        patch.set_edgecolor("#111827")
    for whisker in box["whiskers"]:
        whisker.set_color("#374151")
    for cap in box["caps"]:
        cap.set_color("#374151")

ax.set_xticks(base_positions)
ax.set_xticklabels(PAIR_GROUP_LABELS)
ax.set_ylabel("Cosine similarity")
ax.set_title("Validation image-embedding similarity by pair group")
ax.grid(axis="y", alpha=0.2)
ax.legend(
    handles=[Patch(facecolor=MODEL_COLOURS[label], alpha=0.78, label=MODEL_PLOT_LABELS[label]) for label in MODEL_LABELS],
    loc="lower left",
)
fig.tight_layout()
fig.savefig(BOXPLOT_PATH, dpi=200, bbox_inches="tight")
plt.show()

## 7. Spectra Representation Setup

The image pairwise table above is kept in memory and also written to CSV. The spectra part below uses the selected `Benchtop_N / N/A` absorbance files from the same validation split.


In [ ]:
SELECTED_SPECTRA_MODALITY = "Benchtop_N"
SELECTED_SPECTRA_INSTRUMENT = "N"
SELECTED_SPECTRA_FORMAT = "A"
SELECTED_SPECTRA_LABEL = "Benchtop_N / N/A"
SPECTRA_MODEL_LABEL = "spectra representation"
SPECTRA_ITERATION_LABEL = "Spectra representation"

MODEL_PLOT_LABELS[SPECTRA_MODEL_LABEL] = "Spectra"
MODEL_COLOURS[SPECTRA_MODEL_LABEL] = "#059669"

SPECTRA_PAIRWISE_PATH = ARTIFACTS_DIR / "pairwise_spectra_similarities.csv"
COMBINED_PAIRWISE_PATH = ARTIFACTS_DIR / "test_representation_pairwise_similarities.csv"
COMBINED_SUMMARY_PATH = ARTIFACTS_DIR / "test_representation_similarity_summary.csv"
SPECTRA_BOXPLOT_PATH = ARTIFACTS_DIR / "spectra_similarity_boxplot.png"
COMBINED_BOXPLOT_PATH = ARTIFACTS_DIR / "test_representation_similarity_boxplots.png"

REQUIRED_COMBINED_COLUMNS = [
    "representation",
    "iteration",
    "model",
    "pair_group",
    "item_id_a",
    "item_id_b",
    "fabric_id_a",
    "fabric_id_b",
    "class_a",
    "class_b",
    "cosine_similarity",
]


## 8. Combine Image Pairwise Similarities

The image pairwise similarities were calculated earlier in this notebook. This cell reshapes that image table into the shared schema used for the combined image/spectra comparison.


In [ ]:
def normalise_image_pairwise_for_combined(frame: pd.DataFrame) -> pd.DataFrame:
    output = pd.DataFrame({
        "representation": "image",
        "iteration": frame["iteration"],
        "model": frame["model"],
        "pair_group": frame["pair_group"].astype(str),
        "item_id_a": frame["image_id_a"],
        "item_id_b": frame["image_id_b"],
        "fabric_id_a": frame["fabric_id_a"],
        "fabric_id_b": frame["fabric_id_b"],
        "class_a": frame["class_a"],
        "class_b": frame["class_b"],
        "cosine_similarity": frame["cosine_similarity"].astype(float),
    })
    return output[REQUIRED_COMBINED_COLUMNS]


image_pairwise_for_combined = normalise_image_pairwise_for_combined(pairwise_image_similarities)
image_pair_counts = (
    image_pairwise_for_combined.groupby(["representation", "iteration", "pair_group"], observed=False)
    .size()
    .rename("pairs")
    .reset_index()
)
image_pair_counts

## 9. Build Spectra Vectors

The spectra representation uses the selected `Benchtop_N / N/A` spectra files. The second CSV column is used as the absorbance vector, and each selected spectra record remains a separate item. This keeps within-fabric replicate information available for the pairwise similarity check.

In [ ]:
def eligible_validation_samples() -> pd.DataFrame:
    samples = split_sample_tables["val"].copy()
    eligible = samples[samples["cnn_label"].ne("EXCLUDED_CNN")].copy()
    eligible = eligible.rename(columns={"Sample ID": "sample_id"})
    return eligible[["sample_id", "classification_label", "cnn_label", "split"]].sort_values("sample_id").reset_index(drop=True)


def selected_spectra_rows(fabric_ids: list[str]) -> pd.DataFrame:
    manifest = split_manifest_tables["val"].copy()
    table = manifest[
        manifest["modality_group"].eq("spectra")
        & manifest["modality"].eq(SELECTED_SPECTRA_MODALITY)
        & manifest["instrument"].eq(SELECTED_SPECTRA_INSTRUMENT)
        & manifest["data_format"].eq(SELECTED_SPECTRA_FORMAT)
        & manifest["sample_id"].isin(fabric_ids)
    ].copy()
    missing = sorted(set(fabric_ids) - set(table["sample_id"]))
    if missing:
        raise ValueError(f"Missing selected spectra for validation fabrics: {missing[:5]}")
    return table.sort_values(["sample_id", "replicate", "filename"]).reset_index(drop=True)


def read_spectrum(path: Path) -> tuple[np.ndarray, np.ndarray]:
    array = pd.read_csv(path, header=None).to_numpy(dtype="float32")
    if array.ndim != 2 or array.shape[1] != 2:
        raise ValueError(f"Unexpected spectra shape for {path}: {array.shape}")
    return array[:, 0], array[:, 1]


def build_spectra_items(fabric_table: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    fabric_ids = fabric_table["sample_id"].tolist()
    table = selected_spectra_rows(fabric_ids)
    label_lookup = fabric_table.set_index("sample_id")["classification_label"].to_dict()
    rows = []
    vectors = []
    wavelength_reference = None

    for row_index, row in table.reset_index(drop=True).iterrows():
        wavelengths, values = read_spectrum(Path(row["final_path"]))
        if wavelength_reference is None:
            wavelength_reference = wavelengths
        elif len(wavelengths) != len(wavelength_reference) or not np.allclose(wavelengths, wavelength_reference):
            raise ValueError(f"Wavelength grid mismatch for {row['sample_id']}.")

        replicate = row.get("replicate", row_index)
        filename = row.get("filename", Path(row["final_path"]).name)
        item_id = f"spectra:{row['sample_id']}:replicate:{replicate}:{filename}"
        rows.append({
            "representation": "spectra",
            "model": SPECTRA_MODEL_LABEL,
            "iteration": SPECTRA_ITERATION_LABEL,
            "item_id": item_id,
            "fabric_id": row["sample_id"],
            "class_label": label_lookup[row["sample_id"]],
            "replicate": replicate,
            "filename": filename,
            "source_row": int(row_index),
        })
        vectors.append(values.astype("float32"))

    frame = pd.DataFrame(rows)
    if not frame["item_id"].is_unique:
        frame["item_id"] = frame["item_id"] + ":row:" + frame["source_row"].astype(str)
    return frame, np.vstack(vectors).astype("float32")

In [ ]:
validation_fabrics = eligible_validation_samples()
spectra_rows = selected_spectra_rows(validation_fabrics["sample_id"].tolist())
spectra_item_frame, spectra_matrix = build_spectra_items(validation_fabrics)

records_per_fabric = spectra_item_frame.groupby("fabric_id").size()
spectra_input_summary = pd.DataFrame([
    {
        "representation": "spectra",
        "level": "spectra-record / replicate-level",
        "selected_spectra": SELECTED_SPECTRA_LABEL,
        "records": len(spectra_rows),
        "items": len(spectra_item_frame),
        "fabrics": spectra_item_frame["fabric_id"].nunique(),
        "classes": spectra_item_frame["class_label"].nunique(),
        "vector_dim": spectra_matrix.shape[1],
        "min_records_per_fabric": int(records_per_fabric.min()),
        "max_records_per_fabric": int(records_per_fabric.max()),
        "same_fabric_pairs": "available: different spectra records from the same fabric",
    }
])
spectra_input_summary

## 10. Spectra Pairwise Similarity

Spectra vectors are L2-normalised before cosine similarity. Same-fabric spectra pairs compare different selected spectra records from the same fabric. Spectra cosine values can stay close to 1 because many spectra share a strong global curve shape, so small differences require careful interpretation.

In [ ]:
def l2_normalise(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("Found zero-norm spectra vector.")
    return (matrix / norms).astype("float32")


def pairwise_spectra_similarities(frame: pd.DataFrame, matrix: np.ndarray) -> pd.DataFrame:
    normalised = l2_normalise(matrix)
    row_indices, col_indices = np.triu_indices(len(frame), k=1)
    item_ids = frame["item_id"].to_numpy()
    fabric_ids = frame["fabric_id"].to_numpy()
    labels = frame["class_label"].to_numpy()
    replicates = frame["replicate"].to_numpy()
    filenames = frame["filename"].to_numpy()

    same_fabric = fabric_ids[row_indices] == fabric_ids[col_indices]
    same_class_different_fabric = (~same_fabric) & (labels[row_indices] == labels[col_indices])
    different_class = labels[row_indices] != labels[col_indices]
    pair_group = np.select(
        [same_fabric, same_class_different_fabric, different_class],
        PAIR_GROUP_ORDER,
        default="UNASSIGNED",
    )
    if np.any(pair_group == "UNASSIGNED"):
        raise ValueError("Some spectra pairs were not assigned to a pair group.")

    cosine_similarity = np.sum(normalised[row_indices] * normalised[col_indices], axis=1)
    output = pd.DataFrame({
        "representation": "spectra",
        "iteration": SPECTRA_ITERATION_LABEL,
        "model": SPECTRA_MODEL_LABEL,
        "pair_group": pd.Categorical(pair_group, categories=PAIR_GROUP_ORDER, ordered=True),
        "item_id_a": item_ids[row_indices],
        "item_id_b": item_ids[col_indices],
        "fabric_id_a": fabric_ids[row_indices],
        "fabric_id_b": fabric_ids[col_indices],
        "class_a": labels[row_indices],
        "class_b": labels[col_indices],
        "replicate_a": replicates[row_indices],
        "replicate_b": replicates[col_indices],
        "filename_a": filenames[row_indices],
        "filename_b": filenames[col_indices],
        "cosine_similarity": cosine_similarity.astype(float),
    })
    return output.sort_values(["pair_group", "item_id_a", "item_id_b"]).reset_index(drop=True)


spectra_pairwise = pairwise_spectra_similarities(spectra_item_frame, spectra_matrix)
spectra_pairwise.to_csv(SPECTRA_PAIRWISE_PATH, index=False)

spectra_pair_counts = (
    spectra_pairwise.groupby(["representation", "iteration", "pair_group"], observed=False)
    .size()
    .rename("pairs")
    .reset_index()
)
spectra_pair_counts

## 11. Combined Summary Tables

The image and spectra pairwise tables are combined into one common schema. Both representations now use the same three pair groups, although the image rows are image-level and the spectra rows are spectra-record level.

In [ ]:
def q1(values: pd.Series) -> float:
    return float(values.quantile(0.25))


def q3(values: pd.Series) -> float:
    return float(values.quantile(0.75))


def summary_skeleton(frame: pd.DataFrame) -> pd.DataFrame:
    model_rows = frame[["representation", "iteration", "model"]].drop_duplicates().to_dict("records")
    rows = []
    for model_row in model_rows:
        for pair_group in PAIR_GROUP_ORDER:
            rows.append({**model_row, "pair_group": pair_group})
    return pd.DataFrame(rows)


combined_pairwise = pd.concat(
    [image_pairwise_for_combined, spectra_pairwise[REQUIRED_COMBINED_COLUMNS]],
    ignore_index=True,
)
combined_pairwise["pair_group"] = pd.Categorical(combined_pairwise["pair_group"], categories=PAIR_GROUP_ORDER, ordered=True)
combined_pairwise.to_csv(COMBINED_PAIRWISE_PATH, index=False)

summary_values = (
    combined_pairwise.groupby(["representation", "iteration", "model", "pair_group"], observed=False)["cosine_similarity"]
    .agg(
        number_of_pairs="count",
        mean_cosine_similarity="mean",
        median_cosine_similarity="median",
        min_cosine_similarity="min",
        max_cosine_similarity="max",
        q1_cosine_similarity=q1,
        q3_cosine_similarity=q3,
    )
    .reset_index()
)
combined_summary = summary_skeleton(combined_pairwise).merge(
    summary_values,
    on=["representation", "iteration", "model", "pair_group"],
    how="left",
)
combined_summary["number_of_pairs"] = combined_summary["number_of_pairs"].fillna(0).astype(int)
combined_summary["pair_group"] = pd.Categorical(combined_summary["pair_group"], categories=PAIR_GROUP_ORDER, ordered=True)
combined_summary = combined_summary.sort_values(["representation", "model", "pair_group"]).reset_index(drop=True)
combined_summary.to_csv(COMBINED_SUMMARY_PATH, index=False)
combined_summary

## 12. Spectra Similarity Boxplot

The spectra-only plot includes same-fabric, same-class different-fabric, and different-class spectra-record pairs. The spectra plot uses the observed spectra similarity range.

In [ ]:
spectra_plot_groups = PAIR_GROUP_ORDER
spectra_plot_labels = PAIR_GROUP_LABELS
spectra_values = [
    spectra_pairwise.loc[spectra_pairwise["pair_group"].eq(pair_group), "cosine_similarity"].to_numpy()
    for pair_group in spectra_plot_groups
]

fig, ax = plt.subplots(figsize=(8.5, 4.8))
box = ax.boxplot(
    spectra_values,
    labels=spectra_plot_labels,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "white", "linewidth": 1.5},
)
for patch in box["boxes"]:
    patch.set_facecolor(MODEL_COLOURS[SPECTRA_MODEL_LABEL])
    patch.set_alpha(0.78)
    patch.set_edgecolor("#111827")
for element in ["whiskers", "caps"]:
    for artist in box[element]:
        artist.set_color("#374151")
ax.set_title("Validation spectra similarity by pair group")
ax.set_ylabel("Cosine similarity")
ax.grid(axis="y", alpha=0.2)
all_spectra_values = np.concatenate([values for values in spectra_values if len(values)])
padding = max(0.001, (all_spectra_values.max() - all_spectra_values.min()) * 0.08)
ax.set_ylim(max(0.0, all_spectra_values.min() - padding), min(1.0, all_spectra_values.max() + padding))
fig.tight_layout()
fig.savefig(SPECTRA_BOXPLOT_PATH, dpi=200, bbox_inches="tight")
plt.show()

## 13. Combined Image and Spectra Figure

Panel A shows image-level embeddings from the two CNN iterations. Panel B shows spectra-record level NIR vectors from the selected `Benchtop_N / N/A` representation. The combined figure uses one y-axis range for both panels; the spectra-only figure shows the spectra distribution separately.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), constrained_layout=True)

base_positions = np.arange(len(PAIR_GROUP_ORDER), dtype=float)
offsets = {MODEL_LABELS[0]: -0.18, MODEL_LABELS[1]: 0.18}
for model_label in MODEL_LABELS:
    model_frame = combined_pairwise[
        combined_pairwise["representation"].eq("image") & combined_pairwise["model"].eq(model_label)
    ]
    values = [
        model_frame.loc[model_frame["pair_group"].eq(pair_group), "cosine_similarity"].to_numpy()
        for pair_group in PAIR_GROUP_ORDER
    ]
    box = axes[0].boxplot(
        values,
        positions=base_positions + offsets[model_label],
        widths=0.30,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "white", "linewidth": 1.5},
    )
    for patch in box["boxes"]:
        patch.set_facecolor(MODEL_COLOURS[model_label])
        patch.set_alpha(0.78)
        patch.set_edgecolor("#111827")
    for element in ["whiskers", "caps"]:
        for artist in box[element]:
            artist.set_color("#374151")
axes[0].set_xticks(base_positions)
axes[0].set_xticklabels(PAIR_GROUP_LABELS)
axes[0].set_title("A. Image embeddings")
axes[0].set_ylabel("Cosine similarity")
axes[0].grid(axis="y", alpha=0.2)
axes[0].legend(
    handles=[Patch(facecolor=MODEL_COLOURS[label], alpha=0.78, label=MODEL_PLOT_LABELS[label]) for label in MODEL_LABELS],
    loc="lower left",
)

spectra_positions = np.arange(len(spectra_plot_groups), dtype=float)
box = axes[1].boxplot(
    spectra_values,
    positions=spectra_positions,
    widths=0.45,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "white", "linewidth": 1.5},
)
for patch in box["boxes"]:
    patch.set_facecolor(MODEL_COLOURS[SPECTRA_MODEL_LABEL])
    patch.set_alpha(0.78)
    patch.set_edgecolor("#111827")
for element in ["whiskers", "caps"]:
    for artist in box[element]:
        artist.set_color("#374151")
axes[1].set_xticks(spectra_positions)
axes[1].set_xticklabels(spectra_plot_labels)
axes[1].set_title("B. Spectra representation")
axes[1].set_ylabel("Cosine similarity")
axes[1].grid(axis="y", alpha=0.2)
for ax in axes:
    ax.set_ylim(0.1, 1.02)

fig.savefig(COMBINED_BOXPLOT_PATH, dpi=200, bbox_inches="tight")
plt.show()

## 14. Result Notes

The image and spectra representations are interpreted separately because their cosine-similarity ranges differ. Image similarities cover a wider range, while spectra similarities are concentrated close to 1. A cosine value is therefore interpreted in relation to the representation that produced it.

The expected pattern is that same-fabric pairs have the highest similarity, followed by same-class different-fabric pairs, and then different-class pairs. The plots and summary table show how clearly that pattern appears for the frozen image model, the fine-tuned image model, and the selected spectra representation.
